# 01 - EDA: Sessions and Item Popularity

This notebook focuses on:
- session length distribution
- item popularity distribution
- quick sanity checks for the prepared session data
            


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('default')
DATA_DIR = Path('data/processed')
            


## Load Data

Priority order:
1. `train_sessions.parquet` + `test_sessions.parquet`
2. `yoochoose_sessions.parquet`
            


In [ ]:
train_path = DATA_DIR / 'train_sessions.parquet'
test_path = DATA_DIR / 'test_sessions.parquet'
full_path = DATA_DIR / 'yoochoose_sessions.parquet'

if train_path.exists() and test_path.exists():
    train_df = pd.read_parquet(train_path)
    train_df['split'] = 'train'
    test_df = pd.read_parquet(test_path)
    test_df['split'] = 'test'
    df = pd.concat([train_df, test_df], ignore_index=True)
    source = 'train+test'
elif full_path.exists():
    df = pd.read_parquet(full_path)
    df['split'] = 'full'
    source = 'full'
else:
    raise FileNotFoundError('No processed parquet files found under data/processed')

print(f'data source: {source}')
print(df.head())
print(df.dtypes)
print(f'rows: {len(df):,}')
print(f'sessions: {df["session_id"].nunique():,}')
print(f'items: {df["item_id"].nunique():,}')
            


## Session Lengths


In [ ]:
session_lengths = df.groupby('session_id').size().rename('length')
print(session_lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

plt.figure(figsize=(10, 4))
plt.hist(session_lengths, bins=50)
plt.title('Session Length Distribution')
plt.xlabel('Session length (#items)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.hist(session_lengths, bins=50, log=True)
plt.title('Session Length Distribution (log-y)')
plt.xlabel('Session length (#items)')
plt.ylabel('Count (log scale)')
plt.tight_layout()
plt.show()
            


## Item Popularity


In [ ]:
item_pop = df.groupby('item_id').size().sort_values(ascending=False)
print('Top-20 items by interaction count:')
print(item_pop.head(20))

plt.figure(figsize=(12, 4))
item_pop.head(20).plot(kind='bar')
plt.title('Top-20 Most Popular Items')
plt.xlabel('item_id')
plt.ylabel('interaction count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
item_pop.plot(kind='hist', bins=60, log=True)
plt.title('Item Popularity Distribution')
plt.xlabel('interaction count per item')
plt.ylabel('item frequency (log scale)')
plt.tight_layout()
plt.show()
            


## Quick Split Comparison (if train/test available)


In [ ]:
if 'split' in df.columns and set(df['split'].unique()) >= {'train', 'test'}:
    split_stats = (
        df.groupby('split')
          .agg(
              rows=('item_id', 'size'),
              sessions=('session_id', 'nunique'),
              items=('item_id', 'nunique')
          )
    )
    print(split_stats)

    train_lens = df[df['split'] == 'train'].groupby('session_id').size()
    test_lens = df[df['split'] == 'test'].groupby('session_id').size()

    plt.figure(figsize=(10, 4))
    plt.hist(train_lens, bins=40, alpha=0.6, label='train')
    plt.hist(test_lens, bins=40, alpha=0.6, label='test')
    plt.title('Train vs Test Session Lengths')
    plt.xlabel('session length')
    plt.ylabel('count')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Train/test split files not both present; skipped split comparison.')
            
